In [1]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.append("..")

import pandas as pd
import matplotlib.pyplot as plt

FICHIER_INDICATEURS = "../data/raw/arcep/2025_QoS_indicateurs_globaux.xlsx"
onglets = pd.read_excel(FICHIER_INDICATEURS, sheet_name=None)
for nom, tableau in onglets.items():
    print(nom, tableau.shape)

Notes méthodologiques (9, 36)
Indicateurs globaux (258, 16)
KPI voix-sms par axe transport (868, 16)
KPI data par axe transport (286, 15)


In [2]:
notes = onglets["Notes méthodologiques"]
for valeur in notes.stack():
    if isinstance(valeur, str) and len(valeur) > 200:
        print(valeur, "\n")

Depuis 1997, l’Arcep mène des enquêtes ayant pour objectif d'apprécier, sur une base comparative, la qualité des services mobiles offerts par les opérateurs mobiles. L'enquête vise à être représentative de l'usage des clients des quatre opérateurs de réseaux mobiles, afin de permettre aux utilisateurs de comparer les services de ces opérateurs entre eux.

Pour la 24ème année consécutive, l’Arcep publie les résultats de son enquête d’évaluation de la qualité de service des opérateurs mobiles métropolitains. Plus de trois millions de mesures en 2G, 3G, 4G, 5G, ont été réalisées sur l'ensemble du territoire, dans tous les départements (à l'intérieur et à l'extérieur des bâtiments) et dans les transports (TER, Transiliens, RER, métros, TGV, routes). L’enquête a porté sur les services mobiles les plus répandus : navigation web, lecture de vidéo, transfert de données, SMS et appels vocaux. Les tests réalisés visent à évaluer la performance des réseaux des opérateurs, de manière strictement c

In [3]:
from src.donnees import charger_arcep, fichier_arcep, FICHIERS_ARCEP

arcep = {nom: charger_arcep(fichier_arcep(nom)) for nom in FICHIERS_ARCEP}
for nom, df in arcep.items():
    print(f"{nom:18s} {df.shape[0]:>8} tests  {df.shape[1]:>3} colonnes")

data_habitations     354282 tests   30 colonnes
data_transports      919099 tests   21 colonnes
voix_habitations     223024 tests   33 colonnes
voix_transports      288938 tests   31 colonnes


In [4]:
from src.cibles import ajouter_succes

arcep = {nom: ajouter_succes(df) for nom, df in arcep.items()}

lignes = []
for nom, df in arcep.items():
    for protocole, groupe in df.groupby("protocole"):
        lignes.append({"fichier": nom, "protocole": protocole, "tests": len(groupe),
                       "taux_echec": 1 - groupe["succes"].mean()})
inventaire = pd.DataFrame(lignes).round(3)
inventaire

,fichier,protocole,tests,taux_echec
0,data_habitations,DOWNLOAD,33384,NaN
1,data_habitations,ICMP,17328,0.078
2,data_habitations,STREAM,16933,0.079
3,data_habitations,ULH,16695,0.102
4,data_habitations,UPLOAD,16682,NaN
5,data_habitations,WEB,253260,0.089
6,data_transports,WEB,919099,0.155
7,voix_habitations,OTT,19842,0.211
8,voix_habitations,SMS,101642,0.051
9,voix_habitations,VOIX,101540,0.131


In [5]:
radio = ["rsrp", "rsrq", "rscp", "rx_lev"]
pd.DataFrame({nom: [df[c].notna().mean() if c in df.columns else None for c in radio]
              for nom, df in arcep.items()}, index=radio).round(2)

,data_habitations,data_transports,voix_habitations,voix_transports
rsrp,0.95,0.50,0.95,0.91
rsrq,0.96,NaN,NaN,NaN
rscp,0.02,0.02,0.02,0.04
rx_lev,NaN,NaN,0.00,0.00
